<a href="https://colab.research.google.com/github/dee1empire/-ITAI-1371-ML-Labs-/blob/main/Module_11_Lab_DeloresBledsoe_ITAI_1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# A baseline model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f'Accuracy of baseline Random Forest: {accuracy_baseline:.2%}')

Accuracy of baseline Random Forest: 100.00%


In [8]:
from sklearn.model_selection import GridSearchCV

# Define the grid of hyperparameters to search
# Total combinations: 3 *3 = 9 models to train.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10,None],
    'min_samples_split': [5, 10, 15],
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid, cv=5,
                           scoring='accuracy',
                           n_jobs=-1, verbose=2)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Extract results
best_params_grid = grid_search.best_params_
best_score_grid = grid_search.best_score_

# Evaluate the best model on the test set
best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(X_test)
accuracy_grid = accuracy_score(y_test, y_pred_grid)

# Print the best parameters and the best score
print('The best parameters and the best score')
print('Grid Search Results:')
print(f'Best Hyperparameters found: {best_params_grid}')
print(f'Best Cross-Validation Accuracy: {best_score_grid:.2%}')
print(f'Test Set Accuracy with Best Grid Model: {accuracy_grid:.2%}')


Fitting 5 folds for each of 27 candidates, totalling 135 fits
The best parameters and the best score
Grid Search Results:
Best Hyperparameters found: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}
Best Cross-Validation Accuracy: 94.29%
Test Set Accuracy with Best Grid Model: 100.00%


In [9]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the distribution of hyperparameters to sample from
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [5, 10, 15],
}

# Create a RandomizedSearchCV instance
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    random_state=42
)


# Fit the random search to the data
random_search.fit(X_train, y_train)

# Print the best parameters and the best score
print('Best Parameters found by Random Search: {random_search.best_params_}')
print(f'Best Cross-Validated score: {random_search.best_score_:.2%}')

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters found by Random Search: {random_search.best_params_}
Best Cross-Validated score: 94.29%


In [10]:
# Install AutoGluon
!pip install AutoGluon

import pandas as pd
from autogluon.tabular import TabularPredictor

train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train

test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test

# Create and train the TabularPredictor
# 'time_limit=60' tells it to run for 60 seconds.
predictor = TabularPredictor(label='species', eval_metric='accuracy').fit(train_data=train_data_ag, time_limit=60)

# Evaluate the predictor on the test data
leaderboard = predictor.leaderboard(test_data_ag)
print(leaderboard)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

No path specified. Models will be saved in: "AutogluonModels/ag-20260719_014738"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.29 GB / 12.67 GB (89.1%)
Disk Space Avail:   76.08 GB / 107.72 GB (70.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.

                  model  score_test  score_val eval_metric  pred_time_test  \
0               XGBoost    1.000000   0.952381    accuracy        0.033021   
1      RandomForestEntr    1.000000   0.857143    accuracy        0.090310   
2        ExtraTreesEntr    1.000000   0.904762    accuracy        0.090827   
3        ExtraTreesGini    1.000000   0.904762    accuracy        0.092544   
4      RandomForestGini    1.000000   0.857143    accuracy        0.094701   
5         LightGBMLarge    0.977778   0.857143    accuracy        0.001724   
6              LightGBM    0.955556   0.857143    accuracy        0.001297   
7        NeuralNetTorch    0.955556   0.952381    accuracy        0.009902   
8            LightGBMXT    0.933333   0.952381    accuracy        0.002160   
9              CatBoost    0.933333   0.952381    accuracy        0.004161   
10  WeightedEnsemble_L2    0.933333   0.952381    accuracy        0.004413   

    pred_time_val  fit_time  pred_time_test_marginal  pred_time

            **KNOWLEDGE CHECK**

1. What is the main difference between a model parameter and a hyperparameter?
***Model parameters are internal configuration settings learned directly from the training data during the optimization process. The user does not set them manually.
Examples include the weights and biases in a neural network, or the split thresholds in decision tree.
**With Hyperparameters, these are external settings configured by the programmer before training begins to guide the learning process. The model cannot learn these on its own. (examples: n_estimators, max_depth, and learning rates).


2. When would you choose to use Grid Search over Random Search, and vice-versa?
***To choose Grid Search: Use this when you have a small number of hyperparameters to tune, a small search space, and plenty of computing resources. It is exhaustive and guarantees testing every specified combination, but becomes too computationally expensive as the search space grows. By choosing Random Search, this is used when dealing with a large hyperparameter search space, complex models, or tight computational and time limits. It samples randomly across distributions, usually finding a near-optimal combination much faster and with fewer total model training iterations.


3. Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?
***The top performing model is XGBoost: perfect test score which achieved 100% test accuracy(score_test of 1.000000). It was top-tier generalization which secured the highest validation score of 95.24% (score_val of 0.952381). The effiency leader performed while models like RandomForest also hit 100% test accuracy, XGBoost generalized better during validation and trained extremely fast.
**The reson AutoGluon is so powerful compared to manual tuning is because AutoGluon totally bypasses the trial-and-error nature of manual tuning by introducing automated, advanced machine learning workflows. It introduced multi-framework evaluation that instead of forcing you to manually test different algorithms one-by-one, AutoGluon simultaneously trains completely distinct archetectures like Gradient Boosted Trees(XGBoost, lightGBM,CatBoost), Random Forests, and Deep Learning (NeuralNetTorch). Automatic multi-layer ensembling with AutoGluon excels at "Ensemble Selection" and "Stacking". It automatically takes the predictions of various high-performing base models and feeds them into a secondary meta-model like (WeightedEnsemble_L2). This blends their strengths and significantly improves model robustness. End-to-End Pipeline Automation handles raw data preprocessing, automated feature engineering, missing value imputation, and validation splitting. Therefore, you don't have to perform a manual cleanup code. Then there is time-aware resource management which accepts a hard time constraint(like your time_limit=60 rule). It wisely allocates those 60 seconds to train fast, high-yielding models first, ensuring you get a strong predictive model before your execution budget runs out.

